In [ ]:
from dotenv import load_dotenv
import os
from mcp.server.fastmcp import FastMCP
from tavily import TavilyClient
from typing import Dict, Any
from requests import get

load_dotenv()

mcp = FastMCP(name="LangChain MCP Server")
ClientTavily = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

@mcp.tool()
def web_search(query: str) -> Dict[str, Any]:
    return ClientTavily.search(query=query, num_results=5)

@mcp.resource("urn:github_repo_files")
def github_repo_files() -> Dict[str, Any]:
    url = "https://raw.githubusercontent.com/hemantbyte/GenAI/main/LANGCHAIN/README.md"
    try:
        response = get(url)
        return response.text
    except Exception as e:
        return {"error": f"Error fetching repo files: {str(e)}"}

@mcp.prompt()
def prompt():
    return """
You are a helpful assistant about LangChain, LangGraph and LangSmith.

Use:
- web_search for general web queries.
- github_repo_files to read the GitHub README.

Answer questions clearly.
"""

if __name__ == "__main__":
    print("Starting LangChain MCP server (streamable HTTP)…")
    mcp.run(
        transport="http",     # streamable HTTP
        host="0.0.0.0",        # network bind
        port=8000              # chosen port
    )